# Tours on a layered graph: set-up

The graph-creation part is done here; the algorithms are yours. It mirrors the interactive tutorial (toy OD with six zones, two of them home zones).

**Given (run these cells):**
1. the toy OD table,
2. activity chains and the purpose of each leg,
3. `leg_edges`: the OD rows each leg may use,
4. `build_graph`: the layered graph, with nodes `(layer, zone)`,
5. an optional picture and the arrays for a numpy version.

**Yours (stubs further down):** forward and backward reachability, trimming, counting tours, the completion probability beta, sampling, tour probabilities. A check cell compares each against the numbers shown in the tutorial for home A and the chain Home > Work > Other > Work > Home.

Left out on purpose, as in the tutorial: time periods, distances and capacities.

**Run it with uv** (no project or install needed; from the folder holding this file):

```
uv run --no-project --with jupyterlab --with pandas --with numpy --with networkx --with matplotlib jupyter lab layered_tour_graph.ipynb
```

On a remote machine add `--no-browser --port 8888` and forward the port with `ssh -L 8888:localhost:8888 <host>`; Jupyter prints the URL with its token.

In [ ]:
try:
    import numpy as np
    import pandas as pd
    import networkx as nx
except ImportError as err:
    raise SystemExit(
        f"{err}. Start Jupyter with: uv run --no-project --with jupyterlab --with pandas "
        "--with numpy --with networkx --with matplotlib jupyter lab layered_tour_graph.ipynb"
    )

## 1. The OD

One row per OD pair (an edge of the OD graph). `direction` says whether it leaves home, returns home or is neither. Zones A and B are residential, so tours start and end there.

In [ ]:
ZONES = ["A", "B", "W", "M", "P", "G"]  # Home A, Home B, Work, Mall, Park, Gym
HOMES = ["A", "B"]

od = pd.DataFrame(
    [
        # orig, dest, purpose, direction, trips
        ("A", "W", "HBW", "FromHome", 8), ("A", "M", "HBO", "FromHome", 5), ("A", "P", "HBO", "FromHome", 3),
        ("B", "W", "HBW", "FromHome", 6), ("B", "M", "HBO", "FromHome", 4), ("B", "P", "HBO", "FromHome", 2),
        ("A", "G", "HBO", "FromHome", 2), ("B", "G", "HBO", "FromHome", 1),
        ("W", "A", "HBW", "ToHome", 7), ("M", "A", "HBO", "ToHome", 5), ("P", "A", "HBO", "ToHome", 3),
        ("W", "B", "HBW", "ToHome", 6), ("M", "B", "HBO", "ToHome", 3), ("P", "B", "HBO", "ToHome", 2),
        ("G", "A", "HBO", "ToHome", 2), ("G", "B", "HBO", "ToHome", 1),
        ("W", "M", "NHBW", "NonHome", 4), ("M", "W", "NHBW", "NonHome", 4),
        ("W", "P", "NHBW", "NonHome", 2), ("P", "W", "NHBW", "NonHome", 2), ("W", "W", "NHBW", "NonHome", 1),
        ("W", "G", "NHBW", "NonHome", 2), ("G", "W", "NHBW", "NonHome", 2),
        ("M", "P", "NHBO", "NonHome", 3), ("P", "M", "NHBO", "NonHome", 3), ("M", "M", "NHBO", "NonHome", 2),
        ("M", "G", "NHBO", "NonHome", 2), ("G", "M", "NHBO", "NonHome", 2),
        ("P", "G", "NHBO", "NonHome", 1), ("G", "P", "NHBO", "NonHome", 1),
    ],
    columns=["orig", "dest", "purpose", "direction", "trips"],
)
od.groupby("direction").trips.agg(["count", "sum"])

## 2. Activity chains

A chain such as Home > Work > Other > Work > Home fixes the purpose of every leg. HBW is Home and Work, HBO is Home and Other, NHBW is a trip with a Work end that does not touch Home, NHBO is a trip with neither.

In [ ]:
def purpose_of(a, b):
    if "Home" in (a, b):
        other = b if a == "Home" else a
        return "HB" + other[0]
    return "NHBW" if "Work" in (a, b) else "NHBO"


CHAINS = {
    "H-W-H": ["Home", "Work", "Home"],
    "H-W-O-H": ["Home", "Work", "Other", "Home"],
    "H-W-O-W-H": ["Home", "Work", "Other", "Work", "Home"],
    "H-W-W-H": ["Home", "Work", "Work", "Home"],
    "H-O-O-H": ["Home", "Other", "Other", "Home"],
    "H-W-O-O-H": ["Home", "Work", "Other", "Other", "Home"],
}


def chain_purposes(acts):
    return [purpose_of(a, b) for a, b in zip(acts, acts[1:])]


chain_purposes(CHAINS["H-W-O-W-H"])

## 3. Which OD rows can each leg use?

Leg 1 must be a from-home row leaving the home zone, the last leg a to-home row, the legs in between non-home rows. Every leg also needs the purpose the chain gives it. The result is one DataFrame per leg, keeping the row labels of `od`.

The last leg is not restricted to the home zone, so it also holds rows into the other residential zone. Those lead to a dead end, as in the tutorial.

In [ ]:
def leg_edges(od, home, acts):
    """One DataFrame per leg (index = row label in `od`)."""
    purposes = chain_purposes(acts)
    n = len(purposes)
    legs = []
    for i, p in enumerate(purposes, start=1):
        e = od[(od.purpose == p) & (od.trips > 0)]
        if i == 1:
            e = e[(e.direction == "FromHome") & (e.orig == home)]
        elif i == n:
            e = e[e.direction == "ToHome"]
        else:
            e = e[e.direction == "NonHome"]
        legs.append(e)
    return legs


legs = leg_edges(od, "A", CHAINS["H-W-O-W-H"])
[len(e) for e in legs]

## 4. The layered graph

A tour with n legs gives n + 1 layers. A node is `(layer, zone)`. The edges between layer i-1 and layer i are the rows of leg i, so a zone that is visited twice is two different nodes and no cycles exist. The source is `(0, home)` and the target is `(n, home)`.

Each edge carries `trips`, `purpose`, `leg` and `od_row` (the label in `od`).

In [ ]:
def build_graph(legs, home):
    G = nx.DiGraph()
    n = len(legs)
    source, target = (0, home), (n, home)
    G.add_node(source)
    for i, e in enumerate(legs, start=1):
        for row, r in e.iterrows():
            G.add_edge((i - 1, r.orig), (i, r.dest), trips=int(r.trips), purpose=r.purpose, leg=i, od_row=row)
    G.add_node(target)
    return G, source, target


G, source, target = build_graph(legs, "A")
print("nodes", G.number_of_nodes(), "edges", G.number_of_edges(), "source", source, "target", target)
print("is a DAG:", nx.is_directed_acyclic_graph(G))
for i, e in enumerate(legs, start=1):
    print(f"leg {i} {e.purpose.iloc[0]}:", " ".join(f"{o}->{d}" for o, d in zip(e.orig, e.dest)))

Optional picture (needs matplotlib). Columns are layers, rows are zones; line width grows with trips.

In [ ]:
def draw(G, source=None, target=None, nodes=None, ax=None):
    import matplotlib.pyplot as plt

    pos = {(i, z): (i, -ZONES.index(z)) for i, z in G.nodes}
    ax = ax or plt.subplots(figsize=(9, 4))[1]
    nx.draw_networkx_edges(G, pos, ax=ax, arrows=False, width=[0.5 + 0.4 * G.edges[e]["trips"] ** 0.5 for e in G.edges], alpha=0.6)
    colors = ["tab:green" if n in (source, target) else ("lightgrey" if nodes is not None and n not in nodes else "white") for n in G.nodes]
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors, edgecolors="black", node_size=450)
    nx.draw_networkx_labels(G, pos, labels={n: n[1] for n in G.nodes}, ax=ax, font_size=9)
    ax.set_axis_off()
    return ax


try:
    draw(G, source, target)
except ImportError:
    print("matplotlib is not installed; skipping the picture")

Arrays for a numpy version later: zones become integers 0..5 and every leg becomes three arrays.

In [ ]:
zi = {z: k for k, z in enumerate(ZONES)}


def leg_arrays(legs):
    return [
        (e.orig.map(zi).to_numpy(), e.dest.map(zi).to_numpy(), e.trips.to_numpy(dtype=float))
        for e in legs
    ]


orig, dest, trips = leg_arrays(legs)[1]
orig, dest, trips

---
# Your part

Write these functions. Hints are in the docstrings. Run the check cell at the end; every function reports `todo` until you have written it.

Names: `G` is the full layered graph from `build_graph`. Work with the full graph for the probabilities. The denominators must include the edges that lead nowhere, so do not compute them on the trimmed graph.

In [ ]:
def forward_reachable(G, source):
    """Set of nodes reachable from `source`. A breadth-first search layer by layer."""
    raise NotImplementedError


def backward_reachable(G, target):
    """Set of nodes that can reach `target`. The same search on the reversed graph."""
    raise NotImplementedError


def trim(G, source, target):
    """Subgraph on the nodes found by both searches (and the edges between them)."""
    raise NotImplementedError


def count_paths(Gt, source, target):
    """Number of source-to-target paths of the trimmed graph, without listing them.
    Sweep the layers backwards: ways(target) = 1, ways(node) = sum of ways(next) over its out-edges."""
    raise NotImplementedError


def completion_prob(G, target):
    """beta[node] = probability that a walker at `node` reaches `target`, following the Markov chain
    P(edge) = trips(edge) / total trips of all edges leaving the node (full graph, dead ends included).
    beta[target] = 1, beta of other last-layer nodes = 0, then going backwards:
    beta[node] = sum over out-edges of P(edge) * beta[next]."""
    raise NotImplementedError


def sample_tour(G, beta, source, target, rng, conditioned=True):
    """One walk from `source` as a list of nodes, or None if it does not close at `target`.
    Conditioned: choose an out-edge with weight trips * beta[next]. Plain: weight trips only."""
    raise NotImplementedError


def tour_probabilities(G, beta, source, target):
    """{tuple of zones: probability} for every tour. A tour's probability is the product of P(edge)
    along it divided by beta[source]."""
    raise NotImplementedError

## Checks
Expected values are those of the tutorial.

In [ ]:
def _todo(name, fn):
    try:
        fn()
        print(f"ok    {name}")
    except NotImplementedError:
        print(f"todo  {name}")
    except AssertionError as err:
        print(f"WRONG {name}: {err}")


def run_checks(rng_seed=0):
    G, s, t = build_graph(leg_edges(od, "A", CHAINS["H-W-O-W-H"]), "A")
    assert [len(e) for e in leg_edges(od, "A", CHAINS["H-W-O-W-H"])] == [1, 7, 7, 2]  # the graph itself

    def reach():
        F, B = forward_reachable(G, s), backward_reachable(G, t)
        assert len(F) == 12, f"{len(F)} nodes reached from home, expected 12"
        assert (4, "B") in F and (4, "B") not in B  # arrives at the other residential zone: dead end
        assert len(F & B) == 8, f"{len(F & B)} nodes on a complete tour, expected 8"

    def trimmed():
        Gt = trim(G, s, t)
        assert (Gt.number_of_nodes(), Gt.number_of_edges()) == (8, 10), (Gt.number_of_nodes(), Gt.number_of_edges())

    def counting():
        assert count_paths(trim(G, s, t), s, t) == 4

    def beta_():
        b = completion_prob(G, t)
        want = {(0, "A"): 0.4853, (1, "W"): 0.4853, (2, "M"): 0.5385, (2, "W"): 0.0598, (3, "W"): 0.5385, (4, "A"): 1.0, (4, "B"): 0.0}
        for node, v in want.items():
            assert abs(b[node] - v) < 1e-3, f"beta{node} = {b[node]:.4f}, expected {v}"

    def tours():
        p = tour_probabilities(G, completion_prob(G, t), s, t)
        want = {("A", "W", "M", "W", "A"): 0.4932, ("A", "W", "P", "W", "A"): 0.2466,
                ("A", "W", "G", "W", "A"): 0.2466, ("A", "W", "W", "W", "A"): 0.0137}
        assert set(p) == set(want), sorted(p)
        for k, v in want.items():
            assert abs(p[k] - v) < 1e-3, f"{k}: {p[k]:.4f}, expected {v}"
        assert abs(sum(p.values()) - 1) < 1e-9

    def sampling():
        b = completion_prob(G, t)
        rng = np.random.default_rng(rng_seed)
        exact = tour_probabilities(G, b, s, t)
        draws = [sample_tour(G, b, s, t, rng) for _ in range(20000)]
        assert all(d is not None and d[-1] == t for d in draws), "a conditioned walk must always close"
        freq = pd.Series([tuple(z for _, z in d) for d in draws]).value_counts(normalize=True)
        for k, v in exact.items():
            assert abs(freq.get(k, 0) - v) < 0.015, f"{k}: sampled {freq.get(k, 0):.3f}, exact {v:.3f}"
        plain = [sample_tour(G, b, s, t, rng, conditioned=False) for _ in range(20000)]
        rate = np.mean([d is not None for d in plain])
        assert abs(rate - b[s]) < 0.015, f"plain walk closes {rate:.3f}, beta at the start is {b[s]:.3f}"

    for name, fn in [("reachability", reach), ("trim", trimmed), ("count paths", counting),
                     ("completion probability", beta_), ("tour probabilities", tours), ("sampling", sampling)]:
        _todo(name, fn)


run_checks()

## Other cases to try

| home | chain | tours | beta at the start | tour probabilities |
|---|---|---|---|---|
| A | `H-W-O-H` | 3 | 0.5593 | A,W,M,A 0.4967 · A,W,G,A 0.2649 · A,W,P,A 0.2384 |
| B | `H-W-O-W-H` | 4 | 0.4160 | same four tours as for A, same probabilities |

After that: edit `trips` in the OD and watch the probabilities move, then build the same sweep with the arrays from `leg_arrays`.